# Pre-processing

### 1. Import thư viện cần thiết và load dataset

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

data = pd.read_csv("../data/cleaned/data-cleaned.csv")

display(data)

,seconds_since_midnight,distance_m,signal_mean_pct,signal_std_pct,rx_link_rate_mean,tx_link_rate_mean,rtt_mean_ms,rtt_std_ms,jitter_ms,packet_loss_pct,throughput_down_mbps
0,58515,1,79.416667,0.514929,866.7,866.700000,1.100000,0.316228,0.222222,0.0,94.402610
1,58536,1,80.000000,0.000000,866.7,866.700000,1.800000,2.529822,0.888889,0.0,94.365150
2,58557,1,80.000000,0.000000,866.7,866.700000,2.666667,4.636809,1.750000,10.0,94.672873
3,58583,1,80.000000,0.000000,866.7,866.700000,1.555556,1.333333,1.125000,10.0,93.968420
4,58608,1,81.000000,0.000000,866.7,866.700000,3.800000,4.732864,4.222222,0.0,94.647882
...,...,...,...,...,...,...,...,...,...,...,...
355,33608,15,34.750000,1.544786,1.0,6.500000,459.900000,542.296649,450.777778,0.0,0.104711
356,33645,15,34.000000,1.477098,1.0,7.041667,576.700000,358.224960,375.111111,0.0,0.000000
357,33671,15,34.000000,1.477098,1.0,6.500000,485.700000,481.156258,464.888889,0.0,0.104701
358,33701,15,33.500000,2.504541,1.0,7.583333,192.400000,173.633202,253.444444,0.0,0.000000


### 2. Xóa đi các biến không sử dụng trong mô hình

In [2]:
data = data.drop(columns=['seconds_since_midnight', 'signal_std_pct', 'tx_link_rate_mean', 'rtt_std_ms'])

display(data)

,distance_m,signal_mean_pct,rx_link_rate_mean,rtt_mean_ms,jitter_ms,packet_loss_pct,throughput_down_mbps
0,1,79.416667,866.7,1.100000,0.222222,0.0,94.402610
1,1,80.000000,866.7,1.800000,0.888889,0.0,94.365150
2,1,80.000000,866.7,2.666667,1.750000,10.0,94.672873
3,1,80.000000,866.7,1.555556,1.125000,10.0,93.968420
4,1,81.000000,866.7,3.800000,4.222222,0.0,94.647882
...,...,...,...,...,...,...,...
355,15,34.750000,1.0,459.900000,450.777778,0.0,0.104711
356,15,34.000000,1.0,576.700000,375.111111,0.0,0.000000
357,15,34.000000,1.0,485.700000,464.888889,0.0,0.104701
358,15,33.500000,1.0,192.400000,253.444444,0.0,0.000000


### 3. Chia tập dữ liệu thành 2 phần: 80% để train và 20% để test. Sử dụng `stratify` để đảm bảo các giá trị của biến `distance_m` có tỷ lệ bằng nhau

In [3]:
data_train, data_test = train_test_split(
    data,
    test_size=0.20,
    random_state=911,
    stratify=data["distance_m"]
)

print("Training distance distribution:")
print(data_train["distance_m"].value_counts())

print("\nTesting distance distribution:")
print(data_test["distance_m"].value_counts())

Training distance distribution:
distance_m
5     96
15    96
1     96
Name: count, dtype: int64

Testing distance distribution:
distance_m
5     24
15    24
1     24
Name: count, dtype: int64


### 4. Trong cả 2 tập train và test, điền các ô dữ liệu trống bằng trung bình (mean) của cột dữ liệu tương ứng

In [4]:
columns_to_fill = [
    'rtt_mean_ms',
    'jitter_ms',
    'packet_loss_pct'
]

mean_imputer = SimpleImputer(strategy="mean")

data_train[columns_to_fill] = mean_imputer.fit_transform(data_train[columns_to_fill])
data_test[columns_to_fill] = mean_imputer.fit_transform(data_test[columns_to_fill])

display(data_train[data_train.isna().any(axis=1)])
display(data_test[data_test.isna().any(axis=1)])

,distance_m,signal_mean_pct,rx_link_rate_mean,rtt_mean_ms,jitter_ms,packet_loss_pct,throughput_down_mbps


,distance_m,signal_mean_pct,rx_link_rate_mean,rtt_mean_ms,jitter_ms,packet_loss_pct,throughput_down_mbps


### 5. Lưu lại `data_train`

In [5]:
try:
    data_train.to_csv("../data/preprocessed/data_train.csv", mode="x", index=False)
    print("File created successfully.")
except FileExistsError:
    print("File already exists.")

File already exists.


### 6. Lưu lại `data_test`

In [6]:
try:
    data_test.to_csv("../data/preprocessed/data_test.csv", mode="x", index=False)
    print("File created successfully.")
except FileExistsError:
    print("File already exists.")

File already exists.
